In [ ]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

In [ ]:
%%writefile /kaggle/working/my_agent.py
import hashlib
import logging
import os
import random
import time
import traceback
from collections import defaultdict, deque
from datetime import datetime
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from agents.agent import Agent
from arcengine import FrameData, GameAction, GameState


def setup_experiment_directory(base_output_dir='runs'):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    base_dir = os.path.join(base_output_dir, timestamp)
    os.makedirs(base_dir, exist_ok=True)
    log_file = os.path.join(base_dir, 'logs.log')
    return base_dir, log_file


def get_environment_directory(base_dir, game_id):
    env_dir = os.path.join(base_dir, game_id)
    os.makedirs(env_dir, exist_ok=True)
    return env_dir


def setup_logging_for_experiment(log_file_path):
    root_logger = logging.getLogger()
    for handler in root_logger.handlers[:]:
        if isinstance(handler, logging.FileHandler):
            root_logger.removeHandler(handler)
            handler.close()
    formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
    file_handler = logging.FileHandler(log_file_path, mode="w")
    file_handler.setLevel(root_logger.level)
    file_handler.setFormatter(formatter)
    root_logger.addHandler(file_handler)


class VisualSaliencyNet(nn.Module):
    """Lightweight CNN that scores each (action_type, coord) by expected novelty."""

    def __init__(self, input_channels=16, grid_size=64):
        super().__init__()
        self.grid_size = grid_size
        self.num_action_types = 5

        self.conv1 = nn.Conv2d(input_channels, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.AdaptiveAvgPool2d(8)

        flat = 128 * 8 * 8
        self.action_head = nn.Sequential(
            nn.Linear(flat, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, self.num_action_types),
        )

        self.coord_head = nn.Sequential(
            nn.Linear(flat, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, grid_size * grid_size),
        )

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = self.pool(x).view(x.size(0), -1)
        return self.action_head(x), self.coord_head(x)


class StateGraph:
    """
    Directed graph over observed frames.

    Nodes  : perceptual hash of a frame (uint8 16x64x64 one-hot → md5)
    Edges  : (state_hash, action_idx) → next_state_hash
    Frontier: set of (state_hash, action_idx) pairs never yet tried
    """

    def __init__(self, num_action_types: int, grid_size: int):
        self.num_action_types = num_action_types
        self.grid_size = grid_size
        self.total_actions = num_action_types + grid_size * grid_size

        self.edges: Dict[Tuple[str, int], str] = {}
        self.visit_count: Dict[str, int] = defaultdict(int)
        self.action_visit: Dict[Tuple[str, int], int] = defaultdict(int)
        self.novel_transitions: int = 0

        self.current_state: Optional[str] = None
        self.pending_action: Optional[int] = None

        self.all_states: Dict[str, np.ndarray] = {}

    @staticmethod
    def _hash_frame(frame_np: np.ndarray) -> str:
        return hashlib.md5(frame_np.tobytes()).hexdigest()

    def observe(self, frame_np: np.ndarray) -> str:
        h = self._hash_frame(frame_np)
        if h not in self.all_states:
            self.all_states[h] = frame_np
        self.visit_count[h] += 1

        if self.current_state is not None and self.pending_action is not None:
            key = (self.current_state, self.pending_action)
            if key not in self.edges:
                self.edges[key] = h
                if h != self.current_state:
                    self.novel_transitions += 1

        self.current_state = h
        self.pending_action = None
        return h

    def record_action(self, action_idx: int):
        self.pending_action = action_idx
        if self.current_state is not None:
            self.action_visit[(self.current_state, action_idx)] += 1

    def untried_actions(self, state_hash: str, available_mask: Optional[np.ndarray] = None) -> List[int]:
        tried = {a for (s, a) in self.edges if s == state_hash}
        all_a = list(range(self.total_actions))
        if available_mask is not None:
            all_a = [a for a in all_a if available_mask[a]]
        return [a for a in all_a if a not in tried]

    def ucb_scores(self, state_hash: str, total_steps: int, c: float = 1.5) -> np.ndarray:
        scores = np.zeros(self.total_actions, dtype=np.float32)
        n_s = max(self.visit_count[state_hash], 1)
        for a in range(self.total_actions):
            n_sa = self.action_visit[(state_hash, a)]
            if n_sa == 0:
                scores[a] = float('inf')
            else:
                scores[a] = c * np.sqrt(np.log(n_s) / n_sa)
        return scores

    def bfs_to_frontier(self, start_hash: str, available_mask: Optional[np.ndarray] = None) -> Optional[List[int]]:
        """
        BFS from start_hash through known graph edges to reach a state
        that still has untried actions. Returns action sequence or None.
        """
        if self.untried_actions(start_hash, available_mask):
            return []

        queue = deque([(start_hash, [])])
        visited = {start_hash}

        while queue:
            node, path = queue.popleft()
            if len(path) > 20:
                break
            for (s, a), nxt in self.edges.items():
                if s != node:
                    continue
                if nxt in visited:
                    continue
                new_path = path + [a]
                if self.untried_actions(nxt, available_mask):
                    return new_path
                visited.add(nxt)
                queue.append((nxt, new_path))
        return None

    def reset_level(self):
        self.edges.clear()
        self.visit_count.clear()
        self.action_visit.clear()
        self.novel_transitions = 0
        self.current_state = None
        self.pending_action = None
        self.all_states.clear()


class MyAgent(Agent):

    MAX_ACTIONS = float('inf')
    _MAX_FRAMES = 10

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        seed = int(time.time() * 1000000) + hash(self.game_id) % 1000000
        random.seed(seed)
        np.random.seed(seed % (2**32 - 1))
        torch.manual_seed(seed % (2**32 - 1))
        self.start_time = time.time()

        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"GraphExplorer using device: {self.device}")

        self.base_dir, log_file = setup_experiment_directory()
        setup_logging_for_experiment(log_file)
        env_dir = get_environment_directory(self.base_dir, self.game_id)
        self.logger = logging.getLogger(f"GraphExplorer_{self.game_id}")

        self.grid_size = 64
        self.num_colours = 16
        self.num_action_types = 5
        self.total_actions = self.num_action_types + self.grid_size * self.grid_size

        self.graph = StateGraph(self.num_action_types, self.grid_size)

        self.saliency_net = VisualSaliencyNet(self.num_colours, self.grid_size).to(self.device)
        self.optimizer = optim.Adam(self.saliency_net.parameters(), lr=3e-4)

        self.experience_buffer: deque = deque(maxlen=50000)
        self.batch_size = 32
        self.train_freq = 10

        self.action_list = [
            GameAction.ACTION1, GameAction.ACTION2, GameAction.ACTION3,
            GameAction.ACTION4, GameAction.ACTION5,
        ]

        self.current_level = -1
        self.planned_path: List[int] = []
        self.prev_frame_np: Optional[np.ndarray] = None
        self.prev_action_idx: Optional[int] = None

        self.ucb_c = 2.0
        self.saliency_weight = 0.3
        self.epsilon = 0.05

        self.log_dir = env_dir
        self.logger.info(f"GraphExplorer initialized for game_id: {self.game_id}")

    def append_frame(self, frame: FrameData) -> None:
        self.frames.append(frame)
        if len(self.frames) > self._MAX_FRAMES:
            self.frames = self.frames[-self._MAX_FRAMES:]
        if frame.guid:
            self.guid = frame.guid
        if hasattr(self, "recorder") and not self.is_playback:
            import json
            self.recorder.record(json.loads(frame.model_dump_json()))

    def _get_level(self, frame: FrameData) -> int:
        return getattr(frame, 'score', None) or frame.levels_completed

    def _frame_to_np(self, frame_data: FrameData) -> Optional[np.ndarray]:
        try:
            arr = np.array(frame_data.frame, dtype=np.uint8)
            frame = arr[-1]
            assert frame.shape == (self.grid_size, self.grid_size)
            one_hot = np.zeros((self.num_colours, self.grid_size, self.grid_size), dtype=np.float32)
            for c in range(self.num_colours):
                one_hot[c] = (frame == c)
            return one_hot
        except Exception:
            return None

    def _np_to_tensor(self, frame_np: np.ndarray) -> torch.Tensor:
        return torch.from_numpy(frame_np).unsqueeze(0).to(self.device)

    def _build_available_mask(self, available_actions) -> np.ndarray:
        mask = np.zeros(self.total_actions, dtype=bool)
        has_coord = False
        if available_actions:
            for act in available_actions:
                aid = act.value if hasattr(act, 'value') else int(act)
                if 1 <= aid <= 5:
                    mask[aid - 1] = True
                elif aid == 6:
                    has_coord = True
        else:
            mask[:5] = True
            has_coord = True
        if has_coord:
            mask[5:] = True
        return mask

    def _saliency_scores(self, frame_np: np.ndarray) -> np.ndarray:
        t = self._np_to_tensor(frame_np)
        with torch.no_grad():
            act_logits, coord_logits = self.saliency_net(t)
        act_probs = torch.sigmoid(act_logits).squeeze(0).cpu().numpy()
        coord_probs = torch.sigmoid(coord_logits).squeeze(0).cpu().numpy()
        return np.concatenate([act_probs, coord_probs])

    def _select_action_from_untried(
        self,
        untried: List[int],
        state_hash: str,
        frame_np: np.ndarray,
        mask: np.ndarray,
    ) -> int:
        if random.random() < self.epsilon:
            return random.choice(untried)

        ucb = self.graph.ucb_scores(state_hash, self.action_counter, self.ucb_c)
        sal = self._saliency_scores(frame_np)

        sal_min, sal_max = sal.min(), sal.max()
        if sal_max > sal_min:
            sal = (sal - sal_min) / (sal_max - sal_min)

        scores = ucb + self.saliency_weight * sal
        scores[~mask] = -np.inf

        untried_arr = np.array(untried)
        best = untried_arr[np.argmax(scores[untried_arr])]
        return int(best)

    def _action_idx_to_game_action(self, idx: int, available_actions) -> GameAction:
        if idx < 5:
            act = self.action_list[idx]
            act.reasoning = f"{act.name} [graph-explore]"
            return act
        coord_idx = idx - 5
        y = coord_idx // self.grid_size
        x = coord_idx % self.grid_size
        act = GameAction.ACTION6
        act.set_data({"x": int(x), "y": int(y)})
        act.reasoning = f"ACTION6 at ({x},{y}) [graph-explore]"
        return act

    def _train_saliency_net(self):
        if len(self.experience_buffer) < self.batch_size:
            return
        indices = np.random.choice(len(self.experience_buffer), self.batch_size, replace=False)
        batch = [self.experience_buffer[i] for i in indices]

        states = torch.stack([torch.from_numpy(e['state']).float().to(self.device) for e in batch])
        rewards = torch.tensor([e['reward'] for e in batch], dtype=torch.float32, device=self.device)
        action_indices = torch.tensor([e['action_idx'] for e in batch], dtype=torch.long, device=self.device)

        act_logits, coord_logits = self.saliency_net(states)

        act_targets = action_indices[action_indices < 5]
        act_logits_sel = act_logits[action_indices < 5]
        loss = torch.tensor(0.0, device=self.device, requires_grad=True)

        if len(act_targets) > 0:
            act_rewards = rewards[action_indices < 5]
            gathered = act_logits_sel.gather(1, act_targets.unsqueeze(1)).squeeze(1)
            loss = loss + F.binary_cross_entropy_with_logits(gathered, act_rewards)

        coord_mask = action_indices >= 5
        if coord_mask.any():
            coord_acts = action_indices[coord_mask] - 5
            coord_rewards = rewards[coord_mask]
            coord_logits_sel = coord_logits[coord_mask]
            gathered_c = coord_logits_sel.gather(1, coord_acts.unsqueeze(1)).squeeze(1)
            loss = loss + F.binary_cross_entropy_with_logits(gathered_c, coord_rewards)

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

    def _has_time_elapsed(self) -> bool:
        return (time.time() - self.start_time) >= 8 * 3600 - 5 * 60

    def is_done(self, frames, latest_frame) -> bool:
        try:
            return latest_frame.state is GameState.WIN or self._has_time_elapsed()
        except Exception:
            return True

    def choose_action(self, frames, latest_frame):
        try:
            if self.action_counter == 0:
                print(f"[DEBUG] frame type: {type(latest_frame)}")
                print(f"[DEBUG] state: {latest_frame.state}")
                print(f"[DEBUG] levels_completed: {latest_frame.levels_completed}")
                print(f"[DEBUG] available_actions: {getattr(latest_frame, 'available_actions', 'N/A')}")

            level = self._get_level(latest_frame)
            if level != self.current_level:
                print(f"Level change {self.current_level} → {level} at step {self.action_counter}")
                self.graph.reset_level()
                self.planned_path = []
                self.prev_frame_np = None
                self.prev_action_idx = None
                self.experience_buffer.clear()
                self.saliency_net = VisualSaliencyNet(self.num_colours, self.grid_size).to(self.device)
                self.optimizer = optim.Adam(self.saliency_net.parameters(), lr=3e-4)
                self.current_level = level

            if latest_frame.state in [GameState.NOT_PLAYED, GameState.GAME_OVER]:
                self.graph.reset_level()
                self.planned_path = []
                self.prev_frame_np = None
                act = GameAction.RESET
                act.reasoning = "Resetting game."
                return act

            frame_np = self._frame_to_np(latest_frame)
            if frame_np is None:
                act = random.choice(self.action_list)
                act.reasoning = "Bad frame, random fallback"
                return act

            state_hash = self.graph.observe(frame_np)

            if self.prev_frame_np is not None and self.prev_action_idx is not None:
                prev_hash = StateGraph._hash_frame(self.prev_frame_np)
                changed = (state_hash != prev_hash)
                reward = 1.0 if changed else 0.0
                self.experience_buffer.append({
                    'state': self.prev_frame_np,
                    'action_idx': self.prev_action_idx,
                    'reward': reward,
                })

            available_actions = getattr(latest_frame, 'available_actions', None)
            mask = self._build_available_mask(available_actions)

            if self.planned_path:
                action_idx = self.planned_path.pop(0)
                if not mask[action_idx]:
                    self.planned_path = []
                else:
                    self.graph.record_action(action_idx)
                    self.prev_frame_np = frame_np
                    self.prev_action_idx = action_idx
                    return self._action_idx_to_game_action(action_idx, available_actions)

            untried = self.graph.untried_actions(state_hash, mask)

            if untried:
                action_idx = self._select_action_from_untried(untried, state_hash, frame_np, mask)
            else:
                path = self.graph.bfs_to_frontier(state_hash, mask)
                if path is not None and len(path) > 0:
                    self.planned_path = path[1:]
                    action_idx = path[0]
                else:
                    valid = np.where(mask)[0]
                    if len(valid) == 0:
                        valid = np.arange(self.num_action_types)
                    ucb = self.graph.ucb_scores(state_hash, self.action_counter, self.ucb_c)
                    sal = self._saliency_scores(frame_np)
                    combined = ucb + self.saliency_weight * sal
                    combined[~mask] = -np.inf
                    action_idx = int(np.argmax(combined))

            self.graph.record_action(action_idx)
            self.prev_frame_np = frame_np
            self.prev_action_idx = action_idx

            if self.action_counter % self.train_freq == 0:
                self._train_saliency_net()

            return self._action_idx_to_game_action(action_idx, available_actions)

        except Exception as e:
            print(f"[DEBUG] choose_action CRASHED at step {self.action_counter}: {type(e).__name__}: {e}")
            traceback.print_exc()
            act = random.choice(self.action_list)
            act.reasoning = f"Crash fallback: {e}"
            return act

In [ ]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 \
          --retry-max-time 600 http://gateway:8001/api/games

    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents \
           /kaggle/working/ARC-AGI-3-Agents

    !cp /kaggle/working/my_agent.py \
        /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py

    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write("""from typing import Type, cast
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    "random": Random,
    "myagent": MyAgent,
}
""")

    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write("""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
""")

    !cd /kaggle/working/ARC-AGI-3-Agents && \
        MPLBACKEND=agg \
        python main.py --agent myagent


In [ ]:
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import pandas as pd
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
    submission.head()